<a href="https://colab.research.google.com/github/malikasadnadir-max/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/malikasadnadir-max/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## 1. My lane as an ML task

My lane is **classification**.

The goal is to classify each page as either **declining** or **not declining** based on observable signals such as impressions, content age, days since the last update, average position, CTR, and word count.

Classification fits this task because the outcome has two categories:
- 1 = declining
- 0 = not declining

The prediction can support content-refresh decisions by helping identify pages that may need review.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check the two target classes in the starter data
# Check the two target classes in the starter data

import os
import sys
import subprocess
import pandas as pd

REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (
    df["trend_direction"].str.lower().eq("down").astype(int)
)

print("Task type: Classification")
print("Number of pages:", len(df))
print("Classes:", sorted(df["is_declining_label"].unique()))
print("\nClass distribution:")
print(df["is_declining_label"].value_counts())

Task type: Classification
Number of pages: 30000
Classes: [np.int64(0), np.int64(1)]

Class distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## 2. Target or proxy

My target is `is_declining_label`.

It is created from the observed `trend_direction` field:
- `down` → 1 (declining)
- anything else → 0 (not declining)

This is an observed outcome in the starter data, converted into a binary label for classification.

I would use observable pre-decision signals as model features. I would not use `trend_pct` as a feature because it is directly related to how the declining label is defined and could cause data leakage.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create and inspect the target label

df["is_declining_label"] = (
    df["trend_direction"].str.lower().eq("down").astype(int)
)

print("Target column: is_declining_label")
print("\nTarget counts:")
print(df["is_declining_label"].value_counts())

print("\nTarget rate:")
print(round(df["is_declining_label"].mean(), 3))

Target column: is_declining_label

Target counts:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target rate:
0.542


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 3. Success metric

I will use Precision@50 as the main success metric.

Precision@50 measures the fraction of the 50 pages ranked highest by the model that are actually declining.

This metric fits the decision context because a content team may have limited capacity to review or update only a small number of pages. A good model should therefore place genuinely declining pages near the top of the list.

A useful baseline is the overall declining rate of the dataset, which is 54.2%.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def precision_at_k(y_true, scores, k=50):
    top_k = scores.sort_values(ascending=False).head(k).index
    return y_true.loc[top_k].mean()

baseline_precision = df["is_declining_label"].mean()

print("Baseline declining rate:", round(baseline_precision, 3))
print("Baseline Precision@50:", round(baseline_precision, 3))

Baseline declining rate: 0.542
Baseline Precision@50: 0.542


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## 4. The unit of analysis, as a real dataframe

The unit of analysis is one anonymized webpage/content page.

Each row represents one page that could potentially be reviewed for a content action.

The dataframe contains measurable characteristics of each page, such as content age, update recency, impressions, average search position, click-through rate, word count, and observed trend direction.

The target column `is_declining_label` is created from the observed `trend_direction`.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the unit of analysis as an actual dataframe

columns_to_show = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "trend_direction",
    "is_declining_label"
]

print("One row = one anonymized webpage/content page")
print("Number of rows:", len(df))

df[columns_to_show].head(10)

One row = one anonymized webpage/content page
Number of rows: 30000


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,trend_direction,is_declining_label
0,187,20,3803,10.6,0.76,3221.0,down,1
1,445,25,15320,20.3,0.05,2481.0,down,1
2,141,20,12581,36.5,0.09,3515.0,down,1
3,463,22,11751,6.2,0.49,NaN,stable,0
4,263,14,19140,44.0,0.13,2803.0,down,1
5,147,20,3970,8.5,0.03,3080.0,down,1
6,90,20,20,7.0,0.00,3059.0,down,1
7,445,22,1724,21.2,0.06,NaN,stable,0
8,90,20,32574,46.0,0.09,3807.0,down,1
9,257,104,1240,4.9,0.16,NaN,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML beats a fixed rule here

A fixed rule might use one or two manually chosen thresholds, such as flagging pages that are old and have low impressions.

The pattern may be more complicated because several signals can interact, including content age, update recency, impressions, average position, click-through rate, and word count.

A classification model can learn combinations of these signals instead of relying on one manually chosen threshold.

However, ML does not automatically beat a fixed rule. In my Week 2 experiment, the simple hand rule performed better than the depth-2 decision tree on the tested data. The purpose of ML here is to test whether a more flexible pattern can improve ranking or decision support when evaluated with appropriate validation.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Compare the simple hand rule with the Week 2 baseline results

hand_rule_p20 = 0.900
tree_p20 = 0.550

hand_rule_p50 = 0.680
tree_p50 = 0.600

comparison = pd.DataFrame({
    "Metric": ["Precision@20", "Precision@50"],
    "Hand rule": [hand_rule_p20, hand_rule_p50],
    "Depth-2 tree": [tree_p20, tree_p50]
})

comparison

,Metric,Hand rule,Depth-2 tree
0,Precision@20,0.90,0.55
1,Precision@50,0.68,0.60


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.